<a href="https://colab.research.google.com/github/akshita-singh-2808/Rossmann-Store-Sales/blob/main/notebooks_02_lstm_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

import joblib

In [3]:
project_path = "/content/drive/MyDrive/Rossmann_LSTM_Project"

In [4]:
X_train_seq = np.load(
    f"{project_path}/sequences/X_train_seq.npy"
)

y_train_seq = np.load(
    f"{project_path}/sequences/y_train_seq.npy"
)

X_test_seq = np.load(
    f"{project_path}/sequences/X_test_seq.npy"
)

y_test_seq = np.load(
    f"{project_path}/sequences/y_test_seq.npy"
)

In [5]:
feature_scaler = joblib.load(
    f"{project_path}/models/feature_scaler.pkl"
)

target_scaler = joblib.load(
    f"{project_path}/models/target_scaler.pkl"
)

In [6]:
print(X_train_seq.shape)
print(X_test_seq.shape)

print(y_train_seq.shape)
print(y_test_seq.shape)

(785751, 30, 8)
(58551, 30, 8)
(785751, 1)
(58551, 1)


BUILDING LSTM MODEL

In [7]:
n_timesteps = X_train_seq.shape[1]
n_features = X_train_seq.shape[2]

In [8]:
n_timesteps

30

In [9]:
model=Sequential()

In [10]:
model = Sequential()
model.add(LSTM(128, input_shape=(n_timesteps, n_features),return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(64,return_sequences=False))
model.add(Dropout(0.2))
model.add(Dense(32, activation="relu"))
model.add(Dense(1))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [11]:
model.compile( optimizer="adam", loss="mse", metrics=["mae"])

In [12]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 128)        │        70,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 121,665 (475.25 KB)

 Trainable params: 121,665 (475.25 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
callbacks = [

    EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5
    ),

    ModelCheckpoint(
        filepath=f"{project_path}/models/best_model.keras",
        save_best_only=True
    )

]

In [ ]:
history = model.fit(

    X_train_seq,
    y_train_seq,
    validation_data=(
        X_test_seq,
        y_test_seq
    ),

    epochs=50,
    batch_size=64,
    callbacks=callbacks
)

Epoch 1/50
 1270/12278 ━━━━━━━━━━━━━━━━━━━━ 1:27 8ms/step - loss: 0.0056 - mae: 0.0543